# 04 — Hypothesis tests

*CRISP-DM phase 4 (EDA) · code: `run_statistical_tests.py`*

Formal tests behind the EDA findings, on the validation period 1962–2021 only:

1. Are returns predictable (Ljung-Box), and is volatility (|returns|)?
2. Does today's stress state predict stress in 21 days (χ²)?
3–4. Do the key features differ before stressed / bear days (Mann-Whitney)?
5. Is the validation period comparable to the 2022+ holdout?

← previous: [`03_eda.ipynb`](03_eda.ipynb) · next: [`05_regime_models.ipynb`](05_regime_models.ipynb) →

In [1]:
import sys, warnings
from pathlib import Path

ROOT = Path.cwd().resolve()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent
sys.path[:0] = [str(ROOT / "src"), str(ROOT)]   # project package + root scripts

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import Image, Markdown, display

warnings.filterwarnings("ignore")
pd.set_option("display.max_columns", 40)
pd.set_option("display.width", 160)
plt.rcParams.update({"figure.dpi": 100, "axes.titleweight": "bold", "font.size": 10})

In [2]:
def run_script(module, *args):
    """Run a project script's main() exactly as on the command line, then restore inline plots."""
    import importlib
    mod = importlib.import_module(module)
    old = sys.argv
    sys.argv = [f"{module}.py", *args]
    try:
        code = mod.main()
    finally:
        sys.argv = old
    get_ipython().run_line_magic("matplotlib", "inline")
    return code

In [3]:
run_script("run_statistical_tests")

# Statistical tests and validation vs. holdout

Generated 2026-11-25 18:21 by `run_statistical_tests.py`. Test 1 uses the full history; Tests 2–4 use 1962–2021, the modeling period. Daily observations are autocorrelated and samples are large, so p-values overstate the evidence: read the effect sizes.

## Hypothesis tests

| Test | H0 | n | Statistic | p-value | Effect size | Decision |
|---|---|---|---|---|---|---|
| Ljung–Box, returns | No autocorrelation up to lag 20 | 24,820 | Q(20) = 98.4 | < 0.001 | lag-1 autocorrelation -0.007 | Reject H0 |
| Ljung–Box, absolute returns | No autocorrelation up to lag 20 | 24,820 | Q(20) = 37,890.5 | < 0.001 | lag-1 autocorrelation +0.318 | Reject H0 |
| Chi-square, stressed today vs. in 21 days | Independent | 15,105 | χ²(1) = 3,804.9 | < 0.001 | Cramér's V = 0.50 | Reject H0 |
| Mann–Whitney, 21-day realized vol, by stressed in 21 days | Same distribution | 3,875 vs. 11,230 | U = 3.31e+07 | < 0.001 | r = +0.52 (medians +16.7% vs. +10.8%) | Rejec

0

In [4]:
display(Markdown((ROOT / "reports" / "statistical_tests.md").read_text()))

# Statistical tests and validation vs. holdout

Generated 2026-11-25 18:21 by `run_statistical_tests.py`. Test 1 uses the full history; Tests 2–4 use 1962–2021, the modeling period. Daily observations are autocorrelated and samples are large, so p-values overstate the evidence: read the effect sizes.

## Hypothesis tests

| Test | H0 | n | Statistic | p-value | Effect size | Decision |
|---|---|---|---|---|---|---|
| Ljung–Box, returns | No autocorrelation up to lag 20 | 24,820 | Q(20) = 98.4 | < 0.001 | lag-1 autocorrelation -0.007 | Reject H0 |
| Ljung–Box, absolute returns | No autocorrelation up to lag 20 | 24,820 | Q(20) = 37,890.5 | < 0.001 | lag-1 autocorrelation +0.318 | Reject H0 |
| Chi-square, stressed today vs. in 21 days | Independent | 15,105 | χ²(1) = 3,804.9 | < 0.001 | Cramér's V = 0.50 | Reject H0 |
| Mann–Whitney, 21-day realized vol, by stressed in 21 days | Same distribution | 3,875 vs. 11,230 | U = 3.31e+07 | < 0.001 | r = +0.52 (medians +16.7% vs. +10.8%) | Reject H0 |
| Mann–Whitney, distance from 200-day average, by bear in 21 days | Same distribution | 2,302 vs. 12,803 | U = 6.3e+06 | < 0.001 | r = -0.57 (medians -4.0% vs. +4.8%) | Reject H0 |

Smallest expected count in the chi-square test: 994 (condition: ≥ 5).

Jarque–Bera normality test of daily returns: JB = 361,599, p < 0.001 (skew -0.46, excess kurtosis 18.7).

Contingency table (rows: stressed today; columns: stressed in 21 days):

| | Not stressed in 21 days | Stressed in 21 days |
|---|---|---|
| Not stressed today | 9,795 | 1,435 |
| Stressed today | 1,435 | 2,440 |

## Validation vs. holdout

| Measure | Validation 1962–2021 | Holdout 2022–2026 |
|---|---|---|
| Trading days | 15,105 | 1,207 |
| Annual return / volatility | 7.0% / 16.4% | 9.8% / 17.1% |
| Share of days in a bear market | 15.2% | 16.2% |
| Share of days stressed (main label) | 25.7% | 15.5% |
| Share of days stressed (fixed 25% label) | 7.1% | 10.4% |
| Median 21-day realized volatility | 11.8% | 13.6% |
| Deepest drawdown from a peak | -56.8% | -25.4% |
| Worst day (log return) | -22.9% | -6.2% |

Kolmogorov–Smirnov test on 21-day realized volatility, validation vs. holdout: D = 0.176, p < 0.001.
